# Testeo de funciones y scripts para pipeline descarga de datos 

El objetivo es hacer la descarga de datos y consultar la informacion de un bioproyect en especifico mediante distintas funciones (modularizacion de codigo) que son testeadas en cada celda de este jupiter

In [2]:
#Librerias a utilizar para el ejercicio 
import Bio 
from Bio import Entrez 
import pandas as pd
import argparse
import os, sys
import subprocess
from io import StringIO
from ftplib import FTP
from urllib.parse import urlparse
import urllib.request 
import xml.etree.ElementTree as ET

### summary
Funcion para ir escribiendo el resumen del procesamiento

In [3]:

def summary(text:str, write:str="summary.txt"): 
    """
    Function to write a file with a summary of the search 
    
    Arguments: 
        -text: the information to be written
        -write: the file that would be written 
    
    """ 
    #make sure the path already exist
    direc= os.path.join("output")
    if not os.path.exists(direc):
        os.mkdir(direc) 
    
    #get the path of the file 
    summaryfile=os.path.join(direc,write)
    
    #finaly we write the content we want 
    with open (summaryfile, "a") as sum: 
        sum.write(text)

In [4]:
def make_tsv(dictionary:dict)->str:
    """
    Function that makes a string with the format of a table to be write in a summary.txt 
    Arguments: 
        -dict (dict): dictionary with the information to make the table 
    returns: 
        -str_tsv (str): string with the dicctionary as a table  
    
    """
    str_tsv=""
    
    for key in dictionary.keys():
        str_tsv+=f"{str(key)}\t"
    
    str_tsv+="\n"
    for key in dictionary.keys():
        str_tsv+=f"{str(dictionary[key])}\t" 
        
    return str_tsv


### Linker
Funcion que sirve para hacer la busqueda de las base de datos relacionadas con nuestro proyecto. 
La funcion elink de entrez permite ligar el uid de una base de datos con uids de otras bases de datos permitiendo obtener las relaciones entre bases de datos a traves de los uids ligados, por ejemplo todas las muestras a las que esta ligado un Bioproject, en 
ese caso seria un elink de un uid bioproject contra la base de datos biosample.

In [5]:
def linker(dbs:list,id_uniq:str|list,show=0,db_origin="bioproject")->list: 
    """
    Function that obtain the links with other data bases that are especified by the users 

    Arguments: 
        -dbs: list with the data bases for the consult (list)
        -ID_uniq: the id of the bioporject for search (list)
        -show: flag to know if the consult would be written (int)
        
    Returns:
        -dbs_uids: diccionary with the uids of the data bases linked with exit (dir)
    
    """
    #store the summarize of the consult (numer of uids and database)
    dbs_summarys=""
    #store the overall uids consult as a diccionary 
    dbs_uids={}
    for data_base in dbs: 
        #make the consult 
        try:
            handle = Entrez.elink(dbfrom=db_origin, db=data_base, id=id_uniq)
        except:
            summary(f"It had been imposible make the elink between {id_uniq} bioproject and {data_base}")
        linksBioProj = Entrez.read(handle)
        handle.close()
                
        #Verifying it has link with the data base 
        if linksBioProj[0]["LinkSetDb"]:
            #Obtain de dictionary with the uids in the current data base 
            IDs=linksBioProj[0]["LinkSetDb"][0]["Link"]
            #Create a list with only the uids
            only_ids = [link["Id"] for link in IDs]
            #Get the information as a string for the summary 
            dbs_summarys+=f"The {db_origin} {id_uniq} has {len(IDs)} uids linked with {data_base}\nthe fisrts ones are:{only_ids[:5]}\n"
            #Append all the uids in the diccionari of uids 
            dbs_uids[data_base]=only_ids
        else:
            dbs_summarys+=f"The bioproject {id_uniq} has not link with {data_base}\n"
    
    #write the result of the consult if it has to be in the summary 
    if show:
        
        summary(f"\nThe {db_origin} has this elinks:\n{dbs_summarys}")
    
    
    return [dbs_uids, dbs_summarys]
    

### information 

Funcion que sirve para obtener el resumen de un uid en una base de datos esta sirve para conocer información o metadatos de este uid dentro de una base de datos 

In [6]:
def information(db:str, id:str, info:list=[],show:int=0)->dict: 
    """
    Function that obtain the summary of a consult in a NCBI db according with the info to be specified
    
    Arguments: 
        -db : The data base for the search (str)
        -id : the id for the search (str)
        -info : a list with the info to be displayed, it can be empty for untrimmed consults (list) 
        -show: flag to know if the consult would be written (int)
    Returns: 
        summary_df: a data frame thah sumaraize de consult (data frame)
    
    """
    #Make the consult of the information in the database 
    try:
        handle = Entrez.esummary(db=db, id=id)
    except:
        summary(f"\nIt had been imposible to obtain the {id} summary in the {db} database\n")
        return 0
    proj_summary = Entrez.read(handle) 
    
    #Get only the dictionary with the information 
    try:
        #this for the consults of bioproject and assembly
        consult_dir=proj_summary["DocumentSummarySet"]["DocumentSummary"][0] 
    except: 
        #this for other consults 
        consult_dir=proj_summary[0]
      
    #Obtain the relevant fields in the consult
    if info:
        consult_dir={
            key:consult_dir.get(key) for key in info    
        }
    
    #if the consult is relevant to the summary we write it 
    if show: 
        tsv=make_tsv(consult_dir)
        summary(f"\n{tsv}\n")
    
    #return de diccionary with the information of the consult
    return consult_dir


### Saercher
Esta función servira para reutilizar las busquedas que se efectuen en el script. 
la función esearch de Entrez permite hacer la busqueda de un ID "crudo" dentro de una base de datos, esta es util para obtener los uids dentro de una base de datos a partir de otro un ID: 

Para distintas basese de dartos: 

#### Bioproject 
La busqueda de un esearch dentro de un bioproject sirve para obtener el uid de este proyecto a partir de un id "crudo" como `PRJNA552284` y obtener un uid propio de las bases  

>La importancia de hacer un esearch es que otras funciones de entrez no admiten busquedas con ids crudos 

In [7]:
def searcher(data_base:str, search_term:str)->list: 
    """
    This functios serve as a bridge to link a raw ID to a data base specific ID 
    
    Arguments: 
        -data_base: The name of the data base to be used in the search (str)
        -search_term: the raw id to be search in the data base (str)
    Returns: 
        -ID_uniq: the ID that serves to identify that consult (list)
    """
    
    #Consult the information of the raw ID
    try:    
        handle = Entrez.esearch(db=data_base, term=search_term)
    except: 
        raise ValueError(f"The {search_term} is not yet asociated with a {data_base} UID")
    
    #Obtain the search as a python object, this to be processed 
    search_results = Entrez.read(handle)
    handle.close()  

    #Obtain the list with the ID associated 
    ID_search=search_results["IdList"]

    #Verify the ID is uniq or specify we continue with the first ID in the list 
    number_IDs=len(ID_search)
    
    #we verify it has relation with 
    if number_IDs != 1: 
        if number_IDs < 1:
            summary(F"\nThe {search_term} is no current asociated with a {data_base} uid\n")
            return 0 
        else:
            return ID_search
    else: 
        return ID_search 

### Read XML 
Funcion para leer los sumarrys de los sra UIDs en su apartado "UID"

### Downloading 
This function is to Downoload files prior its UIDs

In [8]:
def download_srr(sra_id,concatenate=None): 
    """
    Function that make de download of SRR files asociated with a sra UID 
    
    Arguments: 
        
    
    """
    
    # Use ecfecth to obtain the information of the UID in sra
    
    handle = Entrez.efetch(db="sra", id=sra_id, rettype="runinfo", retmode="text")
    bite_file= handle.read()
    handle.close()
    
    #Its common this files are binary strings so we decodifie it 
    if isinstance(bite_file, bytes):
        bite_file = bite_file.decode("utf-8")

    #Make a data frame with the string 
    df = pd.read_csv(StringIO(bite_file))
    
    #to debug        
    display(df[['Run','Experiment','Platform','LibraryName','LibraryLayout','Sample','ScientificName','SampleName']].head())
    
    #obtain only the SRR ids related 
    srr_ids=list(df["Run"])
    
    #to debug 
    print(srr_ids)
    
    #The directory for teh output 
    output_dir = "../data"

    for id in srr_ids:
        #Download the files as a srr file
        subprocess.run(["prefetch", "--output-directory", output_dir, id], check=True)
        
        #Transform the files in fastq 
        srr_output_dir = os.path.join(output_dir, id)
        print(srr_output_dir)
        #This line obtain the fastq files separated in case being paired end 
        subprocess.run(["fastq-dump", id, "-O", srr_output_dir,"--split-files", "--gzip"], check=True)
        
        #changue to sumary()
        print(f"The SRRs files {id} were download in {output_dir}/{id}/")

    #if the files belong to a single experiment, the we have to concatenate them 
    if concatenate: 
        #Use _1 to the files fw and _2 to the reversed files 
        #Each one of this list conatin the two path of the fastq files download before 
        files_fw = [output_dir + "/" + f + "/" + f + "_1.fastq.gz" for f in srr_ids]
        files_rv = [output_dir + "/" + f + "/" + f + "_2.fastq.gz" for f in srr_ids]

        # Create the output files with the name of the sample 
        out_fw = output_dir + f"/sample{concatenate}_1.fastq.gz"
        out_rv = output_dir + f"/sample{concatenate}_2.fastq.gz"

    
        # Open the file ../data/id_1_fw/id_2_fw
        with open(out_fw, "wb") as file_out:
            #with zcat we obtain the information of both files in the list we sotre the standar_ouput (files merged)
            p1 = subprocess.Popen(["zcat"] + files_fw, stdout=subprocess.PIPE) 
            #now with gzip we transform the output of zcat in the file with all the runs of a sample
            p2 = subprocess.Popen(["gzip"], stdin=p1.stdout, stdout=file_out)
            #Close the p1 channel
            p1.stdout.close()
            #Make sure all info being correct 
            p2.communicate()

        #We do the same with the rersed data 
        with open(out_rv, "wb") as fout:
            p1 = subprocess.Popen(["zcat"] + files_rv, stdout=subprocess.PIPE)
            p2 = subprocess.Popen(["gzip"], stdin=p1.stdout, stdout=fout)
            p1.stdout.close()
            p2.communicate()
        
        print(f"Files concatenated in {output_dir}/sample{concatenate}_1.fastq.gz")
    else: 
        print(f"Files were no concatenated")
    
    
    
    
        
    
    
    

In [9]:
def reference(organism:str): 
    """
    Function that make the seach of the reference genome and the download if it exits 
    
    Arguments: 
        -organism: the name of the organism of the reference genome (str)
    Retunrs: 
        -1: the correct download of the genome 
        -0: there were an error in the download
    """
    
    #obatin the uids related with the genome of the bioproject organism
    genome_uids=searcher("assembly", organism )

    #list to store the refseq genomes
    assembly_data = []
    #information to the summary consult
    relevant_info=["Organism","RefSeq_category","FtpPath_GenBank"]
    #for al the uids we consult their information
    for uid in genome_uids: 
        #obtain the dictionary with the information 
        uid_summary=information("assembly", uid, relevant_info) 
        #filter only thoes which are refseq
        if uid_summary["RefSeq_category"] != "na": 
            assembly_data.append(uid_summary)


    if not assembly_data: 
        print(f"{organism} does not have a reference genome yet")
        return 0
    else: 
        #we use the fisrt refseq genome in the case it were more than one 
        ftp_link=assembly_data[0]["FtpPath_GenBank"] 
        summary(f"\nThe references genomes were:{assembly_data}\nIt has been used the first one\n")


    
    #We use urllib to parsed the link
    parsed = urlparse(ftp_link)

    #obatain the information of the parsed link  
    ftp_server = parsed.hostname        
    ftp_path = parsed.path            

    summary(f"\nThe FTP_server: {ftp_server}, with the path: {ftp_path}\n")

    # conect to the FTP server in an anonimous form 
    ftp = FTP(ftp_server)
    ftp.login() 
    ftp.cwd(ftp_path)

    #Obtain all the files in the FTP link 
    files = ftp.nlst()  
    #close the ftp link 
    ftp.quit() 
    summary(f"\nThe files stored in the ftp are:\n {files}")

    #Now we filter the files GFF an fasta (fna) does relevant for us 
    fasta_files = [f for f in files if f.endswith("_genomic.fna.gz")]
    gff_files = [f for f in files if f.endswith("_genomic.gff.gz")]

    #Use the same directory of the downloaded SRR
    output_dir = "../data/genome"

    #if the files already exist with continue to the download via urllib.request.urlretrieve
    if fasta_files or gff_files:
        #generate a directory for the genome files 
        os.makedirs(output_dir, exist_ok=True)
        #download the fasta file if it exists 
        if fasta_files:
            #Generate the link related with the fasta file
            fasta_url = ftp_link + "/" + fasta_files[0]
            print("FASTA:", fasta_url)
            #Create the name of the fasta file 
            fasta_out = os.path.join(output_dir, fasta_files[0])
            #Download fasta file 
            urllib.request.urlretrieve(fasta_url, fasta_out)
            print(f"Genome were download with exit in:{fasta_out}")
        else: 
            print(f"There were no fasta files for {organism}")
        #Use the same startegy for the GFF file    
        if gff_files:
            gff_url = ftp_link + "/" + gff_files[0]
            print("GFF:", gff_url)
            gff_out = os.path.join(output_dir,gff_files[0])
            #Download GFF file 
            urllib.request.urlretrieve(gff_url, gff_out)
            print(f"Anotation file GFF were stored in {gff_out}:")
        else:
            print(f"There were no GFF files for {organism}")  
            
    return 1
            

### Obtain srr 
Esta funcion sirve para obtener los ids srr relacionados a muestras de GEO, Biosample o SRA 

In [10]:
def obtain_srr(db:str, db_uids:list)->list | dict: 
    """
    Function to obtain the srr uids associated with the download support data base 
    Arguments:
        -db: database of the UIDs to be linked with the SRR uids(str)
        -db_uids: list with the UIDs to consult (list)
    Returns: 
        -uid_dir: diccionary with the multiples uids of the data base (dict)
    
    """ 
    
    
    match db:
        case "gds": 
            #relevant info for the consults
            relevant_info=['Accession',"entryType","title","summary","taxon","n_samples","FTPLink","suppFile","Samples"]
            #make a dictionary that store all the gsm uid : srr realted uids
            uid_dir={}
            for uid in db_uids: 
                #Get the information of this uid
                consult_dir=information("gds",uid,relevant_info)
                #Get only the sample uid
                samples_ids=[sample['Accession'] for sample in consult_dir.get("Samples",[] )] 
                #Now make the download of ever Sample
                for sample in samples_ids:
                    #consult the sra information 
                    record=searcher("sra",sample)
                    # Make sure it sample has an SRR associated with 
                    if not record["IdList"]:
                        summary(f"They are no srr asociated with {sample} GSM uid")
                    else:  
                        #add to the dictionary
                        uid_dir[uid]=record["IdList"]
            return uid_dir
        
        
        case "sra":
            #the sra uids linked are the srr ids so the return is the same 
            return db_uids
            
        case "biosample": 
            uid_dir={}
            for uid in db_uids:
                #consult the sra asociated with the biosample 
                bio_sample_elink=linker(["sra"],uid,0,"biosample")
                srr_uids=bio_sample_elink[0]["sra"]
                if srr_uids:
                    uid_dir[uid]=srr_uids
                else: 
                    summary(f"They are no srr asociated with {uid} Biosample")
            return uid_dir

### Fetcher 
Funcion que efectua la llamada efetch, la funcion efetch para consultar informacion como texto, tablas o metadatos

In [ ]:
def fetcher(uid:str,db : str ="sra", mode:str = "text" )->pd.DataFrame | int : 
    """
    Function that make the efecth consult and help to obtain metadata of a specified uid in a database
    Arguments: 
        -uid: uniq identifier for the fectch
        -db(str): data base for the fetching
        -mode(str): more or formar to get from the consult
    Returns: 
        -consult_df(data_frame): data frame with the information of the single consult
        -0(int): state of a failed consult
        
    """
    
    #check if we are fetching a sra id
    if db == "sra":
        handle = Entrez.efetch(db= db, id=uid, rettype="runinfo", retmode="text")
        runinfo = handle.read()
        handle.close()

        #make sure we decode de information if it was given in binary format
        if isinstance(runinfo, bytes):
            runinfo = runinfo.decode("utf-8")
            
        #Now we read de csv with pandas
        df = pd.read_csv(StringIO(runinfo))
        #Obtain the relevant information 
        consult_df=df[['Run','Experiment','Platform','LibraryName','LibraryLayout','Sample','ScientificName','SampleName']]
        #finaly we return de data frame with the information 
        return consult_df 
    #if not we asume we are consulting xml information
    try:    
        #make the consult
        handle = Entrez.efetch(db=db, id=uid, retmode=mode)
        xml_text = handle.read()
        handle.close()
        #obtain the codified information and parse as an xml 
        root = ET.fromstring(xml_text)
        #initialice teh dictionary that will store the information 
        consult_df={}
        
        #obtain all the information in the xlm
        for attr in root.findall(".//SAMPLE_ATTRIBUTE"):
            tag = attr.findtext("TAG")
            value = attr.findtext("VALUE")
            #add the tag to the dictionary as it key 
            consult_df[tag]=list(value) 
        #make the dataframe from the dictionary 
        return pd.DataFrame.from_dict(consult_df)
        
    except: 
        return 0
        

## Main 
Esta celda funcionara como nuestro main para ir creando el script 

### Esta celda corresponde a inforamcion y pruebas de babesia

In [12]:
#Main 

Entrez.email= "ismadls@lcg.unam.mx" 

projec_input="PRJNA552284" 

"""
The main function serve as the template for running all the task in the downaload of SRR samples files and reference genome pipeline 
    
"""  
#Obtain all the arguments given from the user
 
organism=""     
#Obtain the UID of the Bioproject only the first One 
    
Bioproject_uid=searcher("bioproject",projec_input)
    
#Verify we are working with only one UID 
if len(Bioproject_uid) != 1: 
    Bioproject_uid=str(Bioproject_uid[0])
    print(f"\n The {projec_input} has more tha one bioprojec UID, it has been used the first one")
else:
    print(f"\nThe uid for the {projec_input} is {Bioproject_uid}\n")


#Obtain the summary of the bioproject (only more relevant features)
relevant_info=['Project_Id','Project_Acc','Project_Data_Type', 'Project_Title', 'Project_Description', 'Organism_Name']
    
Bioproject_summary=information("bioproject",Bioproject_uid, relevant_info,1) 
#Obtain the organism name registered in the NCBI
if not organism:
    organism=Bioproject_summary["Organism_Name"]
    
#Show the bioproject information as a Data frame while the script is running 
summary_df=pd.DataFrame.from_dict(Bioproject_summary, orient="index")
print(summary_df)

dbs=["gds","sra","genome,biosample"]
#obtain the elink information  
Bioproject_elinks=linker(dbs, Bioproject_uid,1) 
#simulamos los inputs del usuario
dbs_interest=["sra"]
sra="all"

for db in dbs_interest:
        #use the string for eval it as a variable for the match in order to obtain the value of the flag 
        match eval(db): 
            #if the user wants to download all the uids associated 
            case "all":
                uids=Bioproject_elinks[0][db]
                srr_ids=obtain_srr(db, uids)
            case None:
                continue
            #if the user has specified a number of uids for downloading each srr asociated 
            case _: 
                uids=Bioproject_elinks[0][db][:int(eval(db))]
                srr_ids=obtain_srr(db,uids)

print(srr_ids)



ref=False
            
if ref:
        #now its time to download the referencie genome 
        if not organism:#If the organism name were no specified 
            #get the biosample uids linked with the Bioproject 
            biosample_elinks=linker(["biosample"],Bioproject_uid)
            #We only select the first one 
            biosample_uid=biosample_elinks[0]["biosample"][0] 
            #Now consult the information in the database
            info_biosample=information("biosample", biosample_uid)
            #Obtain the name of the organism by it first biosample 
            organism=info_biosample["Organism"]
        
  




The uid for the PRJNA552284 is ['552284']

                                                                     0
Project_Id                                                      552284
Project_Acc                                                PRJNA552284
Project_Data_Type                                   Raw sequence reads
Project_Title        Babesia divergens strain:Bd Rouen 1987 | culti...
Project_Description  Babesia divergens is a tick-borne, obligate in...
Organism_Name                                                         
['8459553', '8459552', '8459551', '8459550', '8459549', '8459548']


## Pruebas


In [ ]:
Entrez.email= "ismadls@lcg.unam.mx" 

#ID de proyecto pasilla (si tiene links con GSE)
proyec_input="PRJNA168994" 
data_base="bioproject"

#Searcher

handle = Entrez.esearch(db="bioproject", term=proyec_input)
search_results = Entrez.read(handle)
handle.close()  

print(search_results)

ID_search=search_results["IdList"]

if len(ID_search) != 1: 
        try: 
            ID_uniq=ID_search[0]
        except: 
            raise ValueError(f"The ID {proyec_input} is not associated with any consult at {data_base}") 
        print(f"The ID {proyec_input} is associated with more than one consult, it has been used the firts one")
else: 
    ID_uniq=ID_search[0]
    
ID_uniq
    



{'Count': '1', 'RetMax': '1', 'RetStart': '0', 'IdList': ['168994'], 'TranslationSet': [], 'TranslationStack': [{'Term': 'PRJNA168994[All Fields]', 'Field': 'All Fields', 'Count': '1', 'Explode': 'N'}, 'GROUP'], 'QueryTranslation': 'PRJNA168994[All Fields]'}


'168994'

In [13]:
#Obtener los summarys 

handle = Entrez.esummary(db=data_base, id=ID_uniq)
proj_summary = Entrez.read(handle)

#diccionario para cambiar el nombre de las keys relevantes por su explicación ordinaria
relevant_info={
    'Project_Id':f"ID acces in {data_base}",
    'Project_Acc':f"Raw ID",
    'Project_Data_Type': "Data used", 
    'Project_Title':"Name", 
    'Project_Description':"Description", 
    'Organism_Name':"Organism"
} 
print(relevant_info.get('Project_Id'))

consult_dir=proj_summary["DocumentSummarySet"]["DocumentSummary"][0] 

relevant_things={
    relevant_info[key]:consult_dir.get(key) for key in relevant_info.keys()    
}

display(pd.DataFrame.from_dict(relevant_things, orient="index"))

print(proj_summary["DocumentSummarySet"]["DocumentSummary"][0])

ID acces in bioproject


,0
ID acces in bioproject,168994
Raw ID,PRJNA168994
Data used,Transcriptome or Gene expression
Name,Drosophila melanogaster Transcriptome or Gene ...
Description,RNA profiling data sets generated by the Droso...
Organism,Drosophila melanogaster


DictElement({'TaxId': '7227', 'Project_Id': '168994', 'Project_Acc': 'PRJNA168994', 'Project_Type': 'Primary submission', 'Project_Data_Type': 'Transcriptome or Gene expression', 'Sort_By_ProjectType': '908214', 'Sort_By_DataType': '905574', 'Sort_By_Organism': '133264', 'Project_Subtype': '', 'Project_Target_Scope': 'Multiisolate', 'Project_Target_Material': 'Transcriptome', 'Project_Target_Capture': 'Whole', 'Project_MethodType': 'Array', 'Project_Method': '', 'Project_Objectives_List': [{'Project_ObjectivesType': 'Expression', 'Project_Objectives': ''}], 'Registration_Date': '2012/06/20 00:00', 'Project_Name': 'Drosophila melanogaster', 'Project_Title': 'Drosophila melanogaster Transcriptome or Gene expression', 'Project_Description': 'RNA profiling data sets generated by the Drosophila modENCODE project.', 'Keyword': '', 'Relevance_Agricultural': '', 'Relevance_Medical': '', 'Relevance_Industrial': '', 'Relevance_Environmental': '', 'Relevance_Evolution': '', 'Relevance_Model': 'ye

In [14]:
#linking  


dbs=["gds","sra","genome","biosample"] 


dbs_summarys=""
dbs_UIDs={}
for data_base in dbs: 
    handle = Entrez.elink(dbfrom="bioproject", db=data_base, id=ID_uniq)
    linksBioProj = Entrez.read(handle)
    handle.close()
    
    #se le como una lista el handle 
    
    #Verifying it has link with the data base 
    if linksBioProj[0]["LinkSetDb"]:
        #Obtain de diccionary with the UIDs in the current data base 
        IDs=linksBioProj[0]["LinkSetDb"][0]["Link"]
        #Create a list with only the UIDs
        only_ids = [link["Id"] for link in IDs]
        #Get the information as a string for teh summary 
        dbs_summarys+=f"The bioporject {ID_uniq} has {len(IDs)} UIDs linked with {data_base}\nthe fisrts ones are:{only_ids[:5]}\n "
        #Append all the UIDs in the diccionari of UIDs 
        dbs_UIDs[data_base]=only_ids
    else:
        dbs_summarys+= f"The bioporject {ID_uniq} has not link with {data_base}\n" 

display(dbs_UIDs)
    

print(dbs_summarys)
    

{'gds': ['200040045',
  '200040043',
  '200040042',
  '200040040',
  '200040039',
  '200040038',
  '200040036',
  '200040034',
  '200040016',
  '200040015',
  '200037756',
  '200037440',
  '200037439',
  '200037438',
  '200037437',
  '200037436',
  '200037435',
  '200037434',
  '200037300',
  '200037299',
  '200037297',
  '200037293',
  '200037292',
  '200037291',
  '200025570',
  '200025392',
  '200025390',
  '200025387',
  '200025348',
  '200025347',
  '200024608',
  '200024607',
  '200024605',
  '200024604',
  '200024545',
  '200024544',
  '200024543',
  '200024542',
  '200024541',
  '200024540',
  '200024539',
  '200024317',
  '200024316',
  '200024315',
  '200024314',
  '200024313',
  '200024312',
  '200024311',
  '200024310',
  '200024309',
  '200024308',
  '200024307',
  '200024306',
  '200024305',
  '200024304',
  '200024303',
  '200024302',
  '200024301',
  '200024300',
  '200024299',
  '200024298',
  '200023235',
  '200023234',
  '200023233',
  '200023232',
  '200023231',
  '

The bioporject 168994 has 156 UIDs linked with gds
the fisrts ones are:['200040045', '200040043', '200040042', '200040040', '200040039']
 The bioporject 168994 has 310 UIDs linked with sra
the fisrts ones are:['227117', '214870', '214801', '214798', '214797']
 The bioporject 168994 has 2 UIDs linked with genome
the fisrts ones are:['16743', '47']
 The bioporject 168994 has 370 UIDs linked with biosample
the fisrts ones are:['2197864', '2197863', '2197834', '2197833', '2197832']
 


In [44]:
#ver que devuelve con linker de biosamople
ID_uniq="552284"

biosample_elinks=linker(["biosample"],ID_uniq)

biosmaple_uids=biosample_elinks[0]["biosample"]
biosmaple_uids
conult_test=biosmaple_uids[0]

sra_uids=linker(["sra"],conult_test,0,"biosample")

print(conult_test) 
info_biosample=information("biosample", conult_test)
print(info_biosample["Organism"]
      ) 
search_biosample=searcher("biosample",ID_uniq)

uids= sra_uids[0]["sra"]


#practicando con efetch
df_list=[]
for uid in uids: 
  consult_df=fetcher(uid,"sra")
  df_list.append(consult_df)
  print(list(consult_df.loc[0,:]))

  """
  exp=consult_df.loc[0,'Sample'] 
  print(exp)
  
  consult= Entrez.esummary(db="sra", id=exp)
  srs_con=consult.read()
  handle.close()
  
  print(srs_con)
  
  
  
  handle = Entrez.efetch(db="sra", id=exp, rettype="runinfo", retmode="text")
  #este objeto pued contener un objeto binario b'binary_text'
  runinfo = handle.read()
  handle.close()
  
  print(runinfo)
  #si este objeto es de tipo binario hay que convertirlo a texto para poder leerlo 
  if isinstance(runinfo, bytes):
    runinfo = runinfo.decode("utf-8") 
    print(runinfo)
  
  
  
  root = ET.fromstring(runinfo)

  for attr in root.findall(".//SAMPLE_ATTRIBUTE"):
      tag = attr.findtext("TAG")
      value = attr.findtext("VALUE")
      print(f"{tag}: {value}")
  """
  
  """
  df = pd.read_xml(StringIO(runinfo)) 
  display(df)
  """    

      #print(df.columns)



#obtener informacion de xml  

"""
for uid in uids: 
  print("\n======================================\n")
  info=information("sra", uid)
  print(info["Runs"])
  handle = Entrez.efetch(db="sra", id=uid, retmode="xml")
  xml_text = handle.read()
  handle.close()
  print("\n==================Infromacion de los datos====================\n")
  root = ET.fromstring(xml_text)

  for attr in root.findall(".//SAMPLE_ATTRIBUTE"):
      tag = attr.findtext("TAG")
      value = attr.findtext("VALUE")
      print(f"{tag}: {value}")
"""
all_srrs_df=pd.concat(df_list, ignore_index=True)
display(all_srrs_df)
print(type(all_srrs_df))

12187113
Babesia divergens
['SRR9624152', 'SRX6386485', 'ILLUMINA', 'ML1', 'PAIRED', 'SRS5045988', 'Babesia divergens', 'Bdivergens_ML_PI_RNAseq']
['SRR9624153', 'SRX6386484', 'ILLUMINA', 'ML2', 'PAIRED', 'SRS5045988', 'Babesia divergens', 'Bdivergens_ML_PI_RNAseq']
['SRR9624154', 'SRX6386483', 'ILLUMINA', 'ML3', 'PAIRED', 'SRS5045988', 'Babesia divergens', 'Bdivergens_ML_PI_RNAseq']
['SRR9624155', 'SRX6386482', 'ILLUMINA', 'PI1', 'PAIRED', 'SRS5045988', 'Babesia divergens', 'Bdivergens_ML_PI_RNAseq']
['SRR9624156', 'SRX6386481', 'ILLUMINA', 'PI2', 'PAIRED', 'SRS5045988', 'Babesia divergens', 'Bdivergens_ML_PI_RNAseq']
['SRR9624157', 'SRX6386480', 'ILLUMINA', 'PI3', 'PAIRED', 'SRS5045988', 'Babesia divergens', 'Bdivergens_ML_PI_RNAseq']


,Run,Experiment,Platform,LibraryName,LibraryLayout,Sample,ScientificName,SampleName
0,SRR9624152,SRX6386485,ILLUMINA,ML1,PAIRED,SRS5045988,Babesia divergens,Bdivergens_ML_PI_RNAseq
1,SRR9624153,SRX6386484,ILLUMINA,ML2,PAIRED,SRS5045988,Babesia divergens,Bdivergens_ML_PI_RNAseq
2,SRR9624154,SRX6386483,ILLUMINA,ML3,PAIRED,SRS5045988,Babesia divergens,Bdivergens_ML_PI_RNAseq
3,SRR9624155,SRX6386482,ILLUMINA,PI1,PAIRED,SRS5045988,Babesia divergens,Bdivergens_ML_PI_RNAseq
4,SRR9624156,SRX6386481,ILLUMINA,PI2,PAIRED,SRS5045988,Babesia divergens,Bdivergens_ML_PI_RNAseq
5,SRR9624157,SRX6386480,ILLUMINA,PI3,PAIRED,SRS5045988,Babesia divergens,Bdivergens_ML_PI_RNAseq


<class 'pandas.core.frame.DataFrame'>


In [47]:
prueba_dic={}

str_tsv=""

if isinstance(all_srrs_df, pd.DataFrame):
    columnas=list(all_srrs_df.columns)
    print(columnas)
    
    for col in columnas: 
        str_tsv+=col + "\t" 
    rows=list(all_srrs_df.index)
    str_tsv+="\n"
    for row in rows:
        info_row=list(all_srrs_df.loc[row,:])
        
        for feature in info_row:
            str_tsv+= feature + "\t" 
        str_tsv+="\n" 
    print(str_tsv)

['Run', 'Experiment', 'Platform', 'LibraryName', 'LibraryLayout', 'Sample', 'ScientificName', 'SampleName']
Run	Experiment	Platform	LibraryName	LibraryLayout	Sample	ScientificName	SampleName	
SRR9624152	SRX6386485	ILLUMINA	ML1	PAIRED	SRS5045988	Babesia divergens	Bdivergens_ML_PI_RNAseq	
SRR9624153	SRX6386484	ILLUMINA	ML2	PAIRED	SRS5045988	Babesia divergens	Bdivergens_ML_PI_RNAseq	
SRR9624154	SRX6386483	ILLUMINA	ML3	PAIRED	SRS5045988	Babesia divergens	Bdivergens_ML_PI_RNAseq	
SRR9624155	SRX6386482	ILLUMINA	PI1	PAIRED	SRS5045988	Babesia divergens	Bdivergens_ML_PI_RNAseq	
SRR9624156	SRX6386481	ILLUMINA	PI2	PAIRED	SRS5045988	Babesia divergens	Bdivergens_ML_PI_RNAseq	
SRR9624157	SRX6386480	ILLUMINA	PI3	PAIRED	SRS5045988	Babesia divergens	Bdivergens_ML_PI_RNAseq	

